In [ ]:
##Random Forest Classifier

import spacy

nlp = spacy.load('es')


In [ ]:
import os
from pandas import DataFrame, concat, read_csv
from numpy import isfinite

concat_df_labels = DataFrame()
cat_path = '../data/Categorizacion/'
for i in os.listdir(cat_path):
    print(cat_path + i)
    df_labels = read_csv(cat_path + i, sep=';', error_bad_lines=False)
    
    df_labels = df_labels[(df_labels['QUIEN'].astype(str) != '?')]
    df_labels['QUIEN'] = df_labels['QUIEN'].astype(float)
    
    df_labels['full_text'] = df_labels['full_text'].astype(str)
    
    df_labels['Asesinato'] = df_labels['Asesinato'].apply(lambda x: 1 if x == 'x' or x=='X' else 0)
    df_labels['Violacion'] = df_labels['Violacion'].apply(lambda x: 1 if x == 'x' or x=='X' else 0)
    df_labels['Agresion \nsexual'] = df_labels['Agresion \nsexual'].apply(lambda x: 1 if x == 'x' or x=='X' else 0)
    df_labels['Maltrato'] = df_labels['Maltrato'].apply(lambda x: 1 if x == 'x' or x=='X' else 0)
    df_labels['Acoso'] = df_labels['Acoso'].apply(lambda x: 1 if x == 'x' or x=='X' else 0)
    df_labels['Miedo'] = df_labels['Miedo'].apply(lambda x: 1 if x == 'x' or x=='X' else 0)
    df_labels['Asco\nTristeza\nRabia'] = df_labels['Asco\nTristeza\nRabia'].apply(lambda x: 1 if x == 'x' or x=='X' else 0)
    
    df_labels = df_labels[['id','tweet_id', 'user_name', 'QUIEN', 'Asesinato', 'Violacion', 'Agresion \nsexual', 'Maltrato', 'Acoso', 'Miedo', 'Asco\nTristeza\nRabia', 'full_text']]
    df_labels.set_index('id', inplace=True)
    df_labels = df_labels[isfinite(df_labels['QUIEN'])]
    
    concat_df_labels = concat([concat_df_labels, df_labels], axis=0)

In [ ]:
def keep_meaningful_words(word):
    processed = nlp(word)
    result = [token.lemma_ for token in processed if token.pos_ in ('NOUN', 'VERB', 'ADJ', 'ADV')]
    return ' '.join(result)

random_tweet = df['full_text'].iloc[1000]
print(random_tweet)
print('-' * 30)
print(keep_meaningful_words(random_tweet))

concat_df_labels['filtered_text'] = concat_df_labels['full_text'].apply(keep_meaningful_words)

concat_df_labels.sample()

In [ ]:
text_labels = ['Asesinato', 'Violacion', 'Agresion \nsexual', 'Maltrato', 'Acoso', 'Miedo', 'Asco\nTristeza\nRabia']

In [ ]:
def get_category(x):
    if x['Asesinato']:
        return 0
    elif x['Violacion']:
        return 1
    elif x['Agresion \nsexual']:
        return 2
    elif x['Maltrato']:
        return 3
    elif x['Acoso']:
        return 4
    elif x['Miedo']:
        return 5
    elif x['Asco\nTristeza\nRabia']:
        return 6
    else:
        return 7

concat_df_labels['category'] = concat_df_labels[text_labels].apply(get_category, axis=1)
concat_df_labels['unlabeled'] = concat_df_labels['category'].apply(lambda x: 1 if x == 7 else 0)


In [ ]:
# y : let's make category a number
from sklearn.preprocessing import LabelEncoder
enc = LabelEncoder()
y = enc.fit_transform(concat_df_labels.category)
concat_df_labels['y'] = y

In [ ]:
from pandas import read_pickle

df = read_pickle('../data/cuentalo_json_con_pais_ciudad_limpios.pkl')
df = df[df['tweet_type'] == 'original']

In [ ]:
complex_sentence = df.full_text.values[8888]
processed = nlp(complex_sentence)
for token in processed:
    print(token.text, token.tag_)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

max_jokes_with_that_word = 300
vectorizer30 = CountVectorizer(max_df=max_jokes_with_that_word, max_features=30)
bag_of_words = vectorizer30.fit_transform(df[df['tweet_type'] == 'original'].full_text)
vectorizer30.get_feature_names()

In [ ]:
from pandas import DataFrame
counted = DataFrame(bag_of_words.toarray(), index=df[df['tweet_type'] == 'original'].index, 
                    columns=vectorizer30.get_feature_names())
counted.head(30)

In [ ]:
max_sentences_with_that_word = 300
vectorizer30 = CountVectorizer(max_df=max_sentences_with_that_word, max_features=500)
bag_of_words = vectorizer30.fit_transform(concat_df_labels.filtered_text)

In [ ]:
from sklearn.feature_extraction.text import TfidfTransformer
tfidf_trans = TfidfTransformer()

normalized_bag = tfidf_trans.fit_transform(bag_of_words)

# Display word importance
DataFrame(data=normalized_bag.toarray(), index=concat_df_labels.index, columns=vectorizer30.get_feature_names()).head(25)

In [ ]:
# TfidfVectorizer = CountVectorizer + TFidfTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(max_df=max_jokes_with_that_word, max_features=500)
bag_of_words = tfidf_vectorizer.fit_transform(concat_df_labels.filtered_text)

important_words = tfidf_vectorizer.get_feature_names()
', '.join(important_words)

base_score = concat_df_labels.category.value_counts() / df_labels.shape[0]

In [ ]:
# Split products in train (75%) and test (25%)
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestClassifier
clf = RandomForestClassifier(200)
X = bag_of_words
list_train_scores = []
list_test_scores = []
kf = KFold(n_splits=3, shuffle=True)
for train_index, test_index in kf.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    clf.fit(X_train, y_train)
    base_score = 0.368421
    train_score = clf.score(X_train, y_train)
    test_score = clf.score(X_test, y_test)
    list_train_scores.append(train_score)
    list_test_scores.append(test_score)

    print('Base score (all to "others"): {}'.format(base_score))
    print('Train set score: {}'.format(train_score))
    print('Test set score: {}'.format(test_score) )
    print('*' * 30)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

number_of_trees = 200
clf = RandomForestClassifier(n_estimators=number_of_trees, random_state=1)
clf.fit(X_train, y_train)

base_score = (concat_df_labels.category.value_counts() / concat_df_labels.category.value_counts().sum())[7]
train_score = clf.score(X_train, y_train)
test_score = clf.score(X_test, y_test)

print('Base score (all to "otros"): {}'.format(base_score))
print('Train set score: {}'.format(train_score))
print('Test set score: {}'.format(test_score) )

In [ ]:
import pandas as pd
feature_importances = pd.DataFrame(clf.feature_importances_,
                                   index = vectorizer30.get_feature_names(),
                                    columns=['importance']).sort_values('importance',ascending=False)

In [ ]:
import matplotlib.pyplot as plt

# Plot the feature importances of the forest
plt.figure(figsize=(15, 10))
plt.title("Feature importances")
plt.bar(range(X.shape[1]), importances[indices],
       color="g", yerr=std[indices], align="center")
plt.xticks(range(X.shape[1]), [vectorizer30.get_feature_names()[i] for i in indices], rotation=90)
plt.xlim([-1, 30])
plt.show()

In [ ]:
feature_importances